# 🚛 QuarryVision AI — Analyse Exploratoire des Données (EDA) & Inférence

Ce notebook présente une exploration complète du dataset **quarry-load-verification** (v3) de Roboflow et démontre les résultats d'inférence obtenus avec notre modèle YOLOv8 fine-tuné sur les 7 classes métier de carrière.

In [ ]:
import os
import glob
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from ultralytics import YOLO

DATASET_DIR = Path("../data/quarry_dataset")
MODELS_DIR = Path("../models")
print("Dataset path:", DATASET_DIR.resolve())
print("Model path:", MODELS_DIR.resolve())

## 1. Distribution des Données et Statistiques du Dataset

In [ ]:
splits = ["train", "valid", "test"]
counts = {}
for s in splits:
    imgs = list((DATASET_DIR / s / "images").glob("*.jpg"))
    counts[s] = len(imgs)

df_counts = pd.DataFrame(list(counts.items()), columns=["Split", "Nombre d'images"])
print(df_counts)

plt.figure(figsize=(7, 4))
plt.bar(df_counts["Split"], df_counts["Nombre d'images"], color=["#2e86de", "#27ae60", "#e67e22"])
plt.title("Répartition des images par ensemble")
plt.ylabel("Nombre d'images")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()

## 2. Analyse des Classes Métier et Fréquences d'Annotation

In [ ]:
CLASS_NAMES = {
    0: "BUCKET_CARRY",
    1: "BUCKET_DIG",
    2: "BUCKET_DUMP",
    3: "digger",
    4: "person",
    5: "trailer open bed",
    6: "truck open bed"
}

label_files = list((DATASET_DIR / "train" / "labels").glob("*.txt"))
class_freq = {name: 0 for name in CLASS_NAMES.values()}

for lf in label_files:
    with open(lf, "r") as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                cid = int(parts[0])
                if cid in CLASS_NAMES:
                    class_freq[CLASS_NAMES[cid]] += 1

s_freq = pd.Series(class_freq).sort_values(ascending=False)
print("Fréquence des annotations par classe :\n", s_freq)

plt.figure(figsize=(10, 5))
s_freq.plot(kind="barh", color="#34495e")
plt.title("Nombre d'occurrences des classes métier dans le jeu d'entraînement")
plt.xlabel("Nombre d'annotations")
plt.grid(axis="x", linestyle="--", alpha=0.7)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Inférence Qualitative avec le Modèle Fine-tuné (YOLOv8)

In [ ]:
weights_path = MODELS_DIR / "quarry_load_best.pt"
if weights_path.exists():
    model = YOLO(str(weights_path))
    test_images = list((DATASET_DIR / "test" / "images").glob("*.jpg"))[:3]
    
    for img_p in test_images:
        res = model.predict(str(img_p), conf=0.3)[0]
        annotated_bgr = res.plot()
        annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
        
        plt.figure(figsize=(8, 8))
        plt.imshow(annotated_rgb)
        plt.axis("off")
        plt.title(f"Inférence : {img_p.name}")
        plt.show()
else:
    print("Poids du modèle non trouvés. Lancez l'entraînement via src/train.py")